In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Step 1: Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Step 2: Preprocess data
x_train = x_train[:10000].reshape(10000, -1) / 255.0
y_train = y_train[:10000]

x_test = x_test[:2000].reshape(2000, -1) / 255.0
y_test = y_test[:2000]

# Step 3: Define configurations and activations
configs = [
    (1024, 512, 256),
    (512, 256, 128),
    (256, 128, 64)
]

activations = ["relu", "tanh", "sigmoid"]

results = []

best_accuracy = 0
best_model = None

# Step 4: Train and evaluate models
for activation in activations:
    for config in configs:

        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(3072,)),

            tf.keras.layers.Dense(config[0], activation=activation),
            tf.keras.layers.Dense(config[1], activation=activation),
            tf.keras.layers.Dense(config[2], activation=activation),

            tf.keras.layers.Dense(10, activation="softmax")
        ])

        model.compile(
            optimizer="adam",
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

        model.fit(
            x_train, y_train,
            epochs=10,
            batch_size=128,
            verbose=1
        )

        loss, accuracy = model.evaluate(
            x_test, y_test, verbose=0
        )

        # Store best model
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_model = model

        results.append((activation, config, loss, accuracy))

# Step 5: Display results
for activation, config, loss, accuracy in results:
    print(
        activation,
        config,
        round(loss, 4),
        round(accuracy * 100, 2), "%"
    )

# Step 6: Class names
class_names = [
    "Airplane", "Automobile", "Bird", "Cat", "Deer",
    "Dog", "Frog", "Horse", "Ship", "Truck"
]

# Step 7: Predict a sample image
sample = x_test[0].reshape(1, -1)

pred = best_model.predict(sample, verbose=0)

pred_class = np.argmax(pred)
actual_class = y_test[0][0]

# Step 8: Display image and prediction
plt.imshow(x_test[0].reshape(32, 32, 3))

plt.title(
    "Actual: " + class_names[actual_class] +
    "\nPredicted: " + class_names[pred_class]
)

plt.axis("off")
plt.show()